# 4. Non-Stationary Transformer Playground

In this notebook, we execute the refined, modularized version of the **Non-Stationary Transformer** (NeurIPS 2022). 

Following our MLOps standards, the internal PyTorch mathematics (Series Stationarization and De-stationary Attention) are encapsulated in `src/models/transformer_model.py`. This notebook serves as an interactive entry point to trigger the full pipeline: **Preprocessing -> Cluster Training -> Rolling Forecast -> Evaluation**.

### Pipeline Overview:
1. **Device Detection**: Optimized execution on MPS (Mac), CUDA (Nvidia), or CPU.
2. **Cluster Aggregation**: Training one neural model per behavioral shape (Light, Medium, Heavy, etc.) using hourly averages.
3. **Rolling Window Forecast**: Simulating a day-ahead operational scenario.
4. **Safe Merge 2.0**: Resampling cluster forecasts back to 15-minute intervals and un-scaling via individual client scalers.

In [4]:
import os
import sys
import torch
import pandas as pd

# Add project root to path
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.transformer_model import run_transformer_pipeline
from src.tools.visualization import plot_cluster_portfolio, analyze_time_periods

# ---------------------------------------------------------
# Device selection: MPS (Apple Silicon) > CUDA > CPU
# ---------------------------------------------------------
device = torch.device("mps" if torch.backends.mps.is_available() 
                      else "cuda" if torch.cuda.is_available() 
                      else "cpu")

print(f"Project Root: {PROJECT_ROOT}")
print(f"Active Device: {device.type.upper()}")

Project Root: /Users/federicogiorgi/Library/CloudStorage/OneDrive-Personal/Documenti/GitHub/forecasting-electricity
Active Device: MPS


In [5]:
import os
import sys
import pandas as pd

# 1. Setup Project Root to allow importing from 'src'
# Assuming the notebook is inside the 'notebooks' folder
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 2. Import MLOps functions from our new Transformer module
from src.models.transformer_model import (
    preprocess_and_split,
    train_models,
    predict_models,
    evaluate_models,
    save_transformer_artifacts
)

# 3. Import Visualization tools
from src.tools.visualization import plot_cluster_portfolio, analyze_time_periods

# Define mode and path
MODE = 'day_ahead'
DATA_PATH = os.path.join(PROJECT_ROOT, "Datasets", "processed_electricity_data.parquet")

device = torch.device("mps" if torch.backends.mps.is_available() 
                      else "cuda" if torch.cuda.is_available() 
                      else "cpu")

print(f"Project Root: {PROJECT_ROOT}")
print(f"Active Device: {device.type.upper()}")

# 4. Load the raw parquet data
print("Loading raw dataset...")
df_long = pd.read_parquet(DATA_PATH)
print(f"Data loaded successfully. Shape: {df_long.shape}")

Project Root: /Users/federicogiorgi/Library/CloudStorage/OneDrive-Personal/Documenti/GitHub/forecasting-electricity
Active Device: MPS
Loading raw dataset...
Data loaded successfully. Shape: (41548234, 21)


In [ ]:
# 2. Preprocess, scale, and split the data
print(f"Running preprocessing for mode: {MODE}...")
train_agg, test_agg, test_raw, client_scalers, scaler_weather, regressors = preprocess_and_split(
    df_long, 
    mode=MODE
)

print("\n--- Preprocessing Checks ---")
print(f"Train Aggregated Shape: {train_agg.shape}")
print(f"Test Aggregated Shape: {test_agg.shape}")
print(f"Regressors (Features) Count: {len(regressors)}")
print(f"Regressors List: {regressors}")

Running preprocessing for mode: day_ahead...
Preparing train/test split and scaling (Mode: DAY_AHEAD)...


Scaling Clients:  34%|███▍      | 127/369 [00:02<00:03, 78.40it/s]

In [ ]:
# 3. Train the Transformer models per cluster
print("Starting training process...")
cluster_models = train_models(train_agg, regressors)

print(f"\nTraining complete. Models trained: {list(cluster_models.keys())}")

In [ ]:
# 4. Generate rolling forecasts and un-scale to individual clients
print("Starting prediction and safe merge...")
test_raw = predict_models(
    cluster_models, 
    train_agg, 
    test_agg, 
    test_raw, 
    client_scalers, 
    regressors
)

# CRITICAL CHECK: Verify that the Safe Merge worked and columns exist
print("\n--- Safe Merge Checks ---")
print("Columns in test_raw after prediction:")
print(list(test_raw.columns))

# Let's peek at a few rows that shouldn't be NaN
valid_preds = test_raw.dropna(subset=['Predicted_kW'])
print(f"\nNumber of valid predictions (non-NaN): {len(valid_preds)}")
if len(valid_preds) > 0:
    display(valid_preds[['ClientID', 'Cluster', 'Date', 'Actual_kW', 'Predicted_kW']].head())
else:
    print("WARNING: All predictions are NaN. The merge failed due to date misalignment.")

In [ ]:
# 5. Compute portfolio WMAPE and MAPE
print("Evaluating portfolio metrics...")
portfolio_eval, summary = evaluate_models(test_raw)

# Display the summary table
display(summary)

In [ ]:
# 6. Save models and plot results
print("Saving artifacts to disk...")
save_transformer_artifacts(
    cluster_models, 
    client_scalers, 
    scaler_weather, 
    regressors, 
    mode=MODE
)

print("Generating plots...")
plot_cluster_portfolio(portfolio_eval, summary, model_label="NS-Transformer")
analyze_time_periods(test_raw)

## 1. Run Modular Pipeline

We run the `day_ahead` scenario. The script will:
- Load the processed Parquet dataset.
- Segment and scale consumption at the client level.
- Train 5 neural models (one per cluster).
- Compute WMAPE/MAPE metrics for the entire portfolio.

In [ ]:
DATA_PATH = os.path.join(PROJECT_ROOT, "Datasets", "processed_electricity_data.parquet")

cluster_models, test_raw, portfolio_eval, summary = run_transformer_pipeline(
    DATA_PATH, 
    mode='day_ahead', 
    plot=True
)

Preparing train/test split and scaling (Mode: DAY_AHEAD)...


Scaling Clients: 100%|██████████| 369/369 [00:06<00:00, 54.03it/s]


Aggregating data by Cluster for Transformer training...

Training Cluster 0.0
  Epoch 1 | Train: 0.14016 | Val: 0.36483
  Epoch 2 | Train: 0.03828 | Val: 0.32929
  Epoch 3 | Train: 0.03069 | Val: 0.31711
  Epoch 4 | Train: 0.02662 | Val: 0.30597
  Epoch 5 | Train: 0.02385 | Val: 0.31269
  Epoch 6 | Train: 0.02149 | Val: 0.29741
  Epoch 7 | Train: 0.01986 | Val: 0.29113
  Epoch 8 | Train: 0.01821 | Val: 0.29703
  Epoch 9 | Train: 0.01694 | Val: 0.28405
  Epoch 10 | Train: 0.01590 | Val: 0.29249

Training Cluster 1.0
  Epoch 1 | Train: 0.18916 | Val: 0.08041
  Epoch 2 | Train: 0.06177 | Val: 0.04651
  Epoch 3 | Train: 0.04670 | Val: 0.03685
  Epoch 4 | Train: 0.04009 | Val: 0.03308
  Epoch 5 | Train: 0.03631 | Val: 0.03384
  Epoch 6 | Train: 0.03340 | Val: 0.02918
  Epoch 7 | Train: 0.03131 | Val: 0.02850
  Epoch 8 | Train: 0.02948 | Val: 0.02885
  Epoch 9 | Train: 0.02817 | Val: 0.02687
  Epoch 10 | Train: 0.02690 | Val: 0.02773

Training Cluster 2.0
  Epoch 1 | Train: 0.09527 | Val: 0.

Un-scaling Clients: 100%|██████████| 369/369 [00:00<00:00, 613.91it/s]


Evaluating Portfolio Performance...


KeyError: ['Actual_kW']

## 2. Statistical Deep Dive

Beyond the cluster-level portfolio, we analyze the Transformer's performance across different temporal slices (Weekdays vs Weekends, Peak vs Off-Peak hours).

In [ ]:
analyze_time_periods(test_raw)